Monte Carlo Sensitivity Analysis –  **Bitcoin & Ethereum Baselines (2009–2021)
Fracture Threshold Meter under Realistic Noise**
(5,000 iterations per stage | Gaussian noise: σ=1.0 on D1/G1/F2, σ=0.5 on H3/L2/C2, σ=0.3 on others | Collapse flag: >45%)
This analysis perturbs the consensus scores from each stage to test how stable the Fracture Meter remains under realistic uncertainty in scoring or measurement.
It shows whether small changes push the system across the hinge threshold — and how early (or if) fragility signals appear in durable, long-lived replicators.
Stages tested:

Bitcoin Stage 1 (2009 Genesis)
Bitcoin Stage 3 (2015–2017 Scaling Wars / SegWit)
Ethereum Stage 1 (2015 Genesis)
Ethereum Stage 3 (post-DAO / pre-merge, 2018–2021)

Results summary (to be filled after runs):

Expected: Low collapse risk across all stages under noise (contrast to Terra Stage 3 ~98%)
Key question: Do mature, high-stakes stages (Bitcoin 3, Ethereum 3) remain robust, or do they approach the fracture hinge?

Notebook structure:

Individual cells for each stage baseline + MC run + raw export
Text labels before each block for clarity
All outputs saved as raw_[project]_mc_results_stageX.csv

See individual stage outputs below for mean Fracture Meter, 95% CI, % >45/50/55, and collapse probability.

Bitcoin stage 3 Monty Carlo Run Recalibration  

In [ ]:
import numpy as np

# Bitcoin Stage 3 (2015–2017 Scaling Wars / SegWit)
baseline_bitcoin_stage3 = np.array([
    9.0, 8.7, 8.5, 9.2, 8.5, 8.3, 5.2, 6.8, 3.2, 8.4,
    5.8, 6.3, 8.0, 4.3, 2.2, 5.3, 9.1, 3.3, 4.7, 3.7,
    0.0, 0.0, 0.3, 6.3, 0.0, 2.7, 0.0, -5.7, 3.7, 1.3,
    0.3, -5.3, 0.0, -6.3, 0.0
])


# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

def run_monte_carlo(baseline_array, label):
    fracture_values = []
    collapse_flags = []

    for i in range(n_iter):
        perturbed = baseline_array.copy()

        # Apply noise to all 35 metrics
        perturbed += np.random.normal(0, sigma_others, 35)

        # Apply stronger noise to hinge metrics
        for key, idx in hinge_indices.items():
            if key in ['D1', 'G1', 'F2']:
                perturbed[idx] += np.random.normal(0, sigma_main)
            else:
                perturbed[idx] += np.random.normal(0, sigma_secondary)

        # Clip to valid score range (-10 to +10)
        perturbed = np.clip(perturbed, -10, 10)

        # Extract hinge metrics
        d1 = perturbed[hinge_indices['D1']]
        g1 = perturbed[hinge_indices['G1']]
        f2 = perturbed[hinge_indices['F2']]
        h3 = perturbed[hinge_indices['H3']]
        l2 = perturbed[hinge_indices['L2']]
        c2 = perturbed[hinge_indices['C2']]

        # Compute Fracture Meter
        term1 = d1 * 2
        term2 = (10 - g1) * 2
        term3 = 0.1 * (f2 + h3 + l2 + c2)
        fracture = 15 + term1 + term2 + term3

        fracture_values.append(fracture)

        # Collapse flag
        collapse_flag = 1 if fracture > 45 else 0
        collapse_flags.append(collapse_flag)

    # Summary statistics
    mean_fracture = np.mean(fracture_values)
    median_fracture = np.median(fracture_values)
    ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
    pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
    pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
    pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
    collapse_prob = np.mean(collapse_flags) * 100

    print(f"\n{label} (5,000 iterations):")
    print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
    print(f"Median: {median_fracture:.1f}%")
    print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
    print(f"% >45%: {pct_above_45:.1f}%")
    print(f"% >50%: {pct_above_50:.1f}%")
    print(f"% >55%: {pct_above_55:.1f}%")
    print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

    return fracture_values  # <--- ADD THIS LINE (or add return fracture_values, collapse_flags if you want both)

In [ ]:
import pandas as pd
print("pandas imported successfully as pd")
print("Updated run_monte_carlo function loaded successfully")
print(f"Current n_iter value: {n_iter}")
print(f"hinge_indices keys: {list(hinge_indices.keys())}")

pandas imported successfully as pd
Updated run_monte_carlo function loaded successfully
Current n_iter value: 5000
hinge_indices keys: ['D1', 'G1', 'F2', 'H3', 'L2', 'C2']


In [ ]:
# Bitcoin Stage 3 call - capture fracture_values from the return
fracture_values = run_monte_carlo(baseline_bitcoin_stage3, "Bitcoin Stage 3 (2015–2017 Scaling Wars / SegWit)")

# Now export works
import pandas as pd  # make sure this is run (or add it here if missing)
df_btc_stage3 = pd.DataFrame({'fracture_meter': fracture_values})
df_btc_stage3.to_csv('raw_bitcoin_mc_results_stage3.csv', index=False)
print("Saved: raw_bitcoin_mc_results_stage3.csv")


Bitcoin Stage 3 (2015–2017 Scaling Wars / SegWit) (5,000 iterations):
Mean Fracture Meter: 38.4%
Median: 38.4%
95% CI: 32.6% – 44.1%
% >45%: 1.3%
% >50%: 0.0%
% >55%: 0.0%
Collapse probability (>45% hinge): 1.3%
Saved: raw_bitcoin_mc_results_stage3.csv


Bitcoin Stage 1 MC run

In [ ]:
import numpy as np

# Bitcoin Stage 1 (2009 Genesis)
baseline_bitcoin_stage1 = np.array([
    7.9, 7.2, 7.9, 7.8, 6.3, 6.9, 4.1, 2.1, 1.4, 6.8,
    1.0, 2.7, 6.3, -2.9, 0.4, 3.4, 7.6, -0.8, -1.8, -5.3,
    1.3, 1.7, 1.3, 5.8, -0.8, -2.3, -7.0, -4.7, -1.7, -0.7,
    2.1, -6.7, 0.0, -2.3, -3.3
])


# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

def run_monte_carlo(baseline_array, label):
    fracture_values = []
    collapse_flags = []

    for i in range(n_iter):
        perturbed = baseline_array.copy()

        # Apply noise to all 35 metrics
        perturbed += np.random.normal(0, sigma_others, 35)

        # Apply stronger noise to hinge metrics
        for key, idx in hinge_indices.items():
            if key in ['D1', 'G1', 'F2']:
                perturbed[idx] += np.random.normal(0, sigma_main)
            else:
                perturbed[idx] += np.random.normal(0, sigma_secondary)

        # Clip to valid score range (-10 to +10)
        perturbed = np.clip(perturbed, -10, 10)

        # Extract hinge metrics
        d1 = perturbed[hinge_indices['D1']]
        g1 = perturbed[hinge_indices['G1']]
        f2 = perturbed[hinge_indices['F2']]
        h3 = perturbed[hinge_indices['H3']]
        l2 = perturbed[hinge_indices['L2']]
        c2 = perturbed[hinge_indices['C2']]

        # Compute Fracture Meter
        term1 = d1 * 2
        term2 = (10 - g1) * 2
        term3 = 0.1 * (f2 + h3 + l2 + c2)
        fracture = 15 + term1 + term2 + term3

        fracture_values.append(fracture)

        # Collapse flag
        collapse_flag = 1 if fracture > 45 else 0
        collapse_flags.append(collapse_flag)

    # Summary statistics
    mean_fracture = np.mean(fracture_values)
    median_fracture = np.median(fracture_values)
    ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
    pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
    pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
    pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
    collapse_prob = np.mean(collapse_flags) * 100

    print(f"\n{label} (5,000 iterations):")
    print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
    print(f"Median: {median_fracture:.1f}%")
    print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
    print(f"% >45%: {pct_above_45:.1f}%")
    print(f"% >50%: {pct_above_50:.1f}%")
    print(f"% >55%: {pct_above_55:.1f}%")
    print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

    return fracture_values  # <--- ADD THIS LINE (or add return fracture_values, collapse_flags if you want both)

In [ ]:
import pandas as pd  # safety import

fracture_values = run_monte_carlo(
    baseline_bitcoin_stage1,
    "Bitcoin Stage 1 (2009 Genesis)"
)

df_btc_stage1 = pd.DataFrame({'fracture_meter': fracture_values})
df_btc_stage1.to_csv('raw_bitcoin_mc_results_stage1.csv', index=False)
print("Saved: raw_bitcoin_mc_results_stage1.csv")


Bitcoin Stage 1 (2009 Genesis) (5,000 iterations):
Mean Fracture Meter: 36.8%
Median: 36.9%
95% CI: 31.1% – 42.6%
% >45%: 0.3%
% >50%: 0.0%
% >55%: 0.0%
Collapse probability (>45% hinge): 0.3%
Saved: raw_bitcoin_mc_results_stage1.csv


Ethereum Stage 3 MC run

In [ ]:
import numpy as np

# Ethereum Stage 3 (post-DAO / pre-merge, 2018–2021)
baseline_ethereum_stage3 = np.array([
    8.7, 8.8, 9.1, 3.9, 5.1, 7.6, 7.8, 7.1, 5.9, 8.2,
    5.4, 7.4, 6.8, 4.1, 2.1, 7.4, 8.3, -3.7, -3.4, -3.1,
    0.0, 0.0, 6.3, 6.4, 0.0, -3.9, 0.0, -6.9, -2.4, 0.3,
    3.1, -5.4, 0.0, -7.1, -6.7
])


# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

def run_monte_carlo(baseline_array, label):
    fracture_values = []
    collapse_flags = []

    for i in range(n_iter):
        perturbed = baseline_array.copy()

        # Apply noise to all 35 metrics
        perturbed += np.random.normal(0, sigma_others, 35)

        # Apply stronger noise to hinge metrics
        for key, idx in hinge_indices.items():
            if key in ['D1', 'G1', 'F2']:
                perturbed[idx] += np.random.normal(0, sigma_main)
            else:
                perturbed[idx] += np.random.normal(0, sigma_secondary)

        # Clip to valid score range (-10 to +10)
        perturbed = np.clip(perturbed, -10, 10)

        # Extract hinge metrics
        d1 = perturbed[hinge_indices['D1']]
        g1 = perturbed[hinge_indices['G1']]
        f2 = perturbed[hinge_indices['F2']]
        h3 = perturbed[hinge_indices['H3']]
        l2 = perturbed[hinge_indices['L2']]
        c2 = perturbed[hinge_indices['C2']]

        # Compute Fracture Meter
        term1 = d1 * 2
        term2 = (10 - g1) * 2
        term3 = 0.1 * (f2 + h3 + l2 + c2)
        fracture = 15 + term1 + term2 + term3

        fracture_values.append(fracture)

        # Collapse flag
        collapse_flag = 1 if fracture > 45 else 0
        collapse_flags.append(collapse_flag)

    # Summary statistics
    mean_fracture = np.mean(fracture_values)
    median_fracture = np.median(fracture_values)
    ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
    pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
    pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
    pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
    collapse_prob = np.mean(collapse_flags) * 100

    print(f"\n{label} (5,000 iterations):")
    print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
    print(f"Median: {median_fracture:.1f}%")
    print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
    print(f"% >45%: {pct_above_45:.1f}%")
    print(f"% >50%: {pct_above_50:.1f}%")
    print(f"% >55%: {pct_above_55:.1f}%")
    print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

    return fracture_values  # <--- ADD THIS LINE (or add return fracture_values, collapse_flags if you want both)

In [ ]:
import pandas as pd  # safety import

fracture_values = run_monte_carlo(
    baseline_ethereum_stage3,
    "Ethereum Stage 3 (post-DAO / pre-merge, 2018–2021)"
)

df_eth_stage3 = pd.DataFrame({'fracture_meter': fracture_values})
df_eth_stage3.to_csv('raw_ethereum_mc_results_stage3.csv', index=False)
print("Saved: raw_ethereum_mc_results_stage3.csv")


Ethereum Stage 3 (post-DAO / pre-merge, 2018–2021) (5,000 iterations):
Mean Fracture Meter: 43.8%
Median: 43.9%
95% CI: 38.1% – 49.6%
% >45%: 33.9%
% >50%: 1.8%
% >55%: 0.0%
Collapse probability (>45% hinge): 33.9%
Saved: raw_ethereum_mc_results_stage3.csv




Ethereum stage 1 MC run



In [ ]:
import numpy as np

# Ethereum Stage 1 (Genesis Dateline)
baseline_ethereum_stage1 = np.array([
    8.9, 8.4, 7.6, 2.8, 5.0, 7.8, 7.8, 5.2, 3.4, 7.0,
    2.3, 5.3, 5.3, -2.9, 1.5, 7.0, 8.5, -3.0, -3.0, -4.0,
    0.0, 0.0, 1.7, 4.0, 0.0, -3.0, 0.0, -4.0, 0.7, -1.0,
    3.0, 0.0, 0.0, -5.0, 0.0
])


# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

def run_monte_carlo(baseline_array, label):
    fracture_values = []
    collapse_flags = []

    for i in range(n_iter):
        perturbed = baseline_array.copy()

        # Apply noise to all 35 metrics
        perturbed += np.random.normal(0, sigma_others, 35)

        # Apply stronger noise to hinge metrics
        for key, idx in hinge_indices.items():
            if key in ['D1', 'G1', 'F2']:
                perturbed[idx] += np.random.normal(0, sigma_main)
            else:
                perturbed[idx] += np.random.normal(0, sigma_secondary)

        # Clip to valid score range (-10 to +10)
        perturbed = np.clip(perturbed, -10, 10)

        # Extract hinge metrics
        d1 = perturbed[hinge_indices['D1']]
        g1 = perturbed[hinge_indices['G1']]
        f2 = perturbed[hinge_indices['F2']]
        h3 = perturbed[hinge_indices['H3']]
        l2 = perturbed[hinge_indices['L2']]
        c2 = perturbed[hinge_indices['C2']]

        # Compute Fracture Meter
        term1 = d1 * 2
        term2 = (10 - g1) * 2
        term3 = 0.1 * (f2 + h3 + l2 + c2)
        fracture = 15 + term1 + term2 + term3

        fracture_values.append(fracture)

        # Collapse flag
        collapse_flag = 1 if fracture > 45 else 0
        collapse_flags.append(collapse_flag)

    # Summary statistics
    mean_fracture = np.mean(fracture_values)
    median_fracture = np.median(fracture_values)
    ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
    pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
    pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
    pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
    collapse_prob = np.mean(collapse_flags) * 100

    print(f"\n{label} (5,000 iterations):")
    print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
    print(f"Median: {median_fracture:.1f}%")
    print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
    print(f"% >45%: {pct_above_45:.1f}%")
    print(f"% >50%: {pct_above_50:.1f}%")
    print(f"% >55%: {pct_above_55:.1f}%")
    print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

    return fracture_values  # <--- ADD THIS LINE (or add return fracture_values, collapse_flags if you want both)

In [ ]:
import pandas as pd  # safety import

fracture_values = run_monte_carlo(
    baseline_ethereum_stage1,
    "Ethereum Stage 1 (Genesis)"
)

df_eth_stage1 = pd.DataFrame({'fracture_meter': fracture_values})
df_eth_stage1.to_csv('raw_ethereum_mc_results_stage1.csv', index=False)
print("Saved: raw_ethereum_mc_results_stage1.csv")


Ethereum Stage 1 (Genesis) (5,000 iterations):
Mean Fracture Meter: 39.2%
Median: 39.3%
95% CI: 33.5% – 45.0%
% >45%: 2.5%
% >50%: 0.0%
% >55%: 0.0%
Collapse probability (>45% hinge): 2.5%
Saved: raw_ethereum_mc_results_stage1.csv
